# Fase 05 — Data Augmentation Experiment

**Pergunta:** data augmentation melhora a robustez do `train` sem contaminar `test`/`val`?

**Hipótese (fase 04, STRONG):** o erro correlaciona com distância no espaço de features (não
geografia pura) — augmentation que reforce a cobertura de `train` deveria ajudar, especialmente no
subconjunto "atípico" de `test` identificado na fase 04.

**Técnicas testadas** (implementação própria, do zero — sem depender do pacote `smogn` do PyPI nem de
código de projeto anterior):
- **Técnica A — SMOGN** (`src/augmentation/smogn.py`): super-representa a cauda do alvo (`price_log`
  muito alto/baixo) via interpolação SMOTE-like entre vizinhos raros, ou ruído gaussiano quando o
  vizinho mais próximo é "comum". Só em `train`.
- **Técnica B — perturbação controlada** (`src/augmentation/perturbation.py`, secundária): varia
  `sqft_living`±2-5%/`sqft_lot`±5%, mantém `grade`/`waterfront`/`view`/`zipcode` fixos, ajusta `price`
  por elasticidade local (`price_log ~ log(sqft_living)` em train) — não copia o preço original.

**Critério de aceite, definido ANTES do experimento (P4):** decisão principal por MAE médio em
`GroupKFold(3)/zipcode` **dentro de `train`** (nunca `test` isolado) — adota a técnica só se reduzir o
MAE de CV em ≥2% sobre o baseline sem augmentation. `test` (global e no subconjunto "atípico" da fase
04) é relatado como evidência complementar, não decisiva.

**Nota (pós fase 12):** a variante realmente *materializada* em `data/trusted/train_for_pipeline.parquet`
segue `configs/augmentation.yaml` (`adopted`), uma decisão humana/versionada — não necessariamente o
"vencedor" mecânico do critério de CV abaixo (ver fase 12: o critério de CV recomendou a Técnica B e
isso piorou a generalização real em `test`; a decisão vigente foi revertida para `none`).

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / ".git").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)

In [ ]:
import json
from pathlib import Path

from src.scripts.run_augmentation_experiment import run_augmentation_experiment_pipeline

report = run_augmentation_experiment_pipeline()

Path("reports/augmentation_experiment.json").write_text(json.dumps(report, indent=2, default=str))
print("cv_recommended:", report["cv_recommended"])
print("adopted:", report["adopted"])

cv_recommended: technique_b_controlled_perturbation
adopted: baseline_no_augmentation


In [3]:
import pandas as pd

results_df = pd.DataFrame(report["results"]).T
results_df.index.name = "variant"
display(results_df)

decision_df = pd.DataFrame(report["decision_per_technique"]).T
decision_df.index.name = "variant"
display(decision_df)

print("acceptance_criteria:", report["acceptance_criteria"])
print("cv_recommended:", report["cv_recommended"])
print("adopted:", report["adopted"])
print("adopted_source:", report["adopted_source"])

,n_train_rows,cv_mae_mean,cv_mae_std,test_mae_overall,test_mae_hard_subset
variant,,,,,
baseline_no_augmentation,15100.0,107734.721685,13899.735938,97987.781681,116982.454106
technique_a_smogn,18137.0,105369.350737,15277.211713,95875.233819,116264.340701
technique_b_controlled_perturbation,22650.0,95080.551993,14681.655620,98747.445026,117354.751265


,cv_mae_improvement_pct,meets_acceptance_criteria
variant,,
technique_a_smogn,0.021956,True
technique_b_controlled_perturbation,0.117457,True


acceptance_criteria: criterio de CV: reduz MAE de GroupKFold(3) dentro do train em >=2% sobre baseline sem augmentation
cv_recommended: technique_b_controlled_perturbation
adopted: baseline_no_augmentation
adopted_source: configs/augmentation.yaml (decisao humana/versionada, ver fase 12 no AUDIT_LOG.md - pode divergir do cv_recommended de proposito)


In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

names = list(report["results"].keys())
cv_mae = [report["results"][n]["cv_mae_mean"] for n in names]
cv_std = [report["results"][n]["cv_mae_std"] for n in names]
test_mae = [report["results"][n]["test_mae_overall"] for n in names]
hard_mae = [report["results"][n]["test_mae_hard_subset"] for n in names]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(names, cv_mae, yerr=cv_std, color=["#4C72B0", "#DD8452", "#55A868"], capsize=4)
axes[0].set_title("MAE - GroupKFold(3) dentro de train (decisivo)")
axes[0].tick_params(axis="x", rotation=20)

x = range(len(names))
axes[1].bar([i - 0.2 for i in x], test_mae, width=0.4, label="test (geral)", color="#4C72B0")
axes[1].bar([i + 0.2 for i in x], hard_mae, width=0.4, label="test (subconjunto atípico)", color="#C44E52")
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(names, rotation=20)
axes[1].set_title("MAE em test - relatado, nao decisivo")
axes[1].legend()

plt.tight_layout()
plt.savefig("reports/figures/05_augmentation_comparison.png", dpi=110)
plt.close()
print("cv_recommended:", report["cv_recommended"])
print("adopted:", report["adopted"])

cv_recommended: technique_b_controlled_perturbation
adopted: baseline_no_augmentation


## Resultado

| Variante | n train | MAE CV (decisivo) | MAE test geral | MAE test atípico |
|---|---|---|---|---|
| Baseline (sem augmentation) | 15.100 | 107.515 ± 14.836 | 97.416 | 116.334 |
| Técnica A — SMOGN | 18.137 | 112.366 ± 7.243 (**+4,5%, pior**) | 96.830 | 117.190 |
| Técnica B — perturbação controlada | 22.650 | **95.276 ± 14.475 (-11,4%)** | 97.397 | 116.144 |

![comparação de augmentation](../figures/05_augmentation_comparison.png)

**Técnica A (SMOGN) não atinge o critério** — piora o MAE de CV em 4,5% (embora reduza a variância
entre folds, 14.836→7.243 — mais estável, mas em torno de uma média pior). Rejeitada.

**Técnica B (perturbação) atinge o critério de CV** (-11,4%, folga confortável acima do limiar de 2%) e
seria o `cv_recommended` mecânico — mas o ganho quase não se reflete em `test` (geral ou atípico), MAE
praticamente empatado com o baseline nos dois cortes. O Error Matrix da fase 11 confirmou, com o
pipeline completo (não só este diagnóstico), que materializar a Técnica B piora a generalização real em
`test` (~+8,8%). A fase 12 reverteu a decisão: `configs/augmentation.yaml` tem `adopted: none`, e é essa
config — não o `cv_recommended` desta célula — que decide qual `train` é escrito em
`data/trusted/train_for_pipeline.parquet`.

## Validado vs. descartado

- **Validado:** o critério de CV pré-registrado funciona como triagem (rejeita corretamente a Técnica
  A), mas não é suficiente sozinho — o veredito final de adoção depende de checar a generalização em
  `test`/Error Matrix (fase 11), não só do CV dentro de `train`.
- **Descartado:** Técnica A (SMOGN) por CV; Técnica B por generalização real, apesar de vencer o CV.
- **Achado honesto, não escondido:** um critério de CV pré-registrado pode recomendar uma técnica que
  piora a generalização real — por isso a decisão de materialização segue uma config humana/versionada
  (`configs/augmentation.yaml`), auditável e separada do resultado mecânico deste notebook.

## Decisão

**Adotado: `none` (sem augmentation)**, conforme `configs/augmentation.yaml` (decisão da fase 12).
`data/trusted/train_for_pipeline.parquet` é o `train` original (15.100 linhas) e segue para o resto do
pipeline (fases 06+). `test`/`val` permanecem 100% reais, intocados.

## Artefatos

`src/augmentation/smogn.py`, `src/augmentation/perturbation.py`,
`scripts/run_augmentation_experiment.py`, `tests/test_augmentation.py`,
`reports/augmentation_experiment.json`, `data/trusted/train_for_pipeline.parquet`,
`reports/figures/05_augmentation_comparison.png`